# Train the pneumonia detector on Google Colab

Runtime → Change runtime type → **T4 GPU**, then run all cells. Training takes roughly 10–15 minutes.

You need a Kaggle API token (`kaggle.json`, from Kaggle → Settings → API → *Create New Token*).

In [ ]:
!git clone https://github.com/Abhhiiissshhek/pneumonia-detection.git
%cd pneumonia-detection
!pip install -q -e . kaggle

## 1. Download the dataset
Upload `kaggle.json` when prompted.

In [ ]:
from google.colab import files
import os
if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    files.upload()  # choose kaggle.json
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!bash scripts/download_data.sh

## 2. Train
Fine-tunes the last ResNet block and the classifier, with class-weighted loss and early stopping.

In [ ]:
!python -m pneumonia.train --epochs 10 --batch-size 64 --num-workers 2

## 3. Evaluate on the held-out test set

In [ ]:
!python -m pneumonia.evaluate

from IPython.display import Image, display
display(Image('reports/confusion_matrix.png'), Image('reports/roc_curve.png'))

## 4. Grad-CAM on a test image

In [ ]:
import glob
img = sorted(glob.glob('data/chest_xray/test/PNEUMONIA/*'))[0]
!python -m pneumonia.predict "$img" --gradcam reports/gradcam_example.png
display(Image('reports/gradcam_example.png', width=400))

## 5. Download the weights and reports
Upload `pneumonia_resnet18.pt` as an asset of a **GitHub Release** on your repo. The Streamlit app downloads it automatically from there.
Copy the report images into `docs/` and the numbers from `reports/metrics.json` into the README.

In [ ]:
!zip -qr results.zip models/pneumonia_resnet18.pt reports/
files.download('results.zip')